# NoClip · 실험 결과 비교 이미지 (Google Colab)

드라이브에 올려 둔 PLY 들을 **같은 카메라 위치·각도**에서 렌더링해서 한 장에 나란히 놓음. 발표 스크린샷용.

- 카메라: 그 구간의 COLMAP 촬영 위치 중 몇 곳 (`2_정합데이터/4차/<구간>/sparse`)
- 시점마다: 정면 · 왼쪽 45° · 오른쪽 70° · 천장(위 35°) · 바닥(아래 35°) · 옆으로 이동(촬영 경로 밖, VR 처럼)
- 배경은 분홍 (= 비어서 뒤가 보이는 곳)
- 결과: 드라이브 `3DGS/3_학습결과/비교이미지/` 에 `구간_카메라.png` (행 = 시점, 열 = 버전) + `tiles/` 낱장

**실행**: 런타임 유형 T4 GPU → 모두 실행. 처음엔 gsplat 컴파일 10~30분, 렌더링 자체는 몇 분.

## 1. 설정

In [ ]:
import os, subprocess
if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
    raise RuntimeError("GPU 런타임이 아님 -> 런타임 > 런타임 유형 변경 에서 T4 선택")
from google.colab import drive
drive.mount('/content/drive')

D = "/content/drive/MyDrive/3DGS"
SECTIONS = [1]            # 비교할 구간. 예: [1, 2, 3, 4, 5, 6]
N_CAMS   = 4              # 구간마다 고를 촬영 위치 수
WIDTH    = 640            # 낱장 폭 (px)

# 버전 이름 -> PLY 경로 ({n} = 구간 번호). 파일이 없는 버전은 자동으로 건너뜀
VERSIONS = {
    "기본 Brush":        D + "/3_학습결과/4차_Brush_기본/{n}.ply",
    "E 설정":            D + "/3_학습결과/4차_Brush_E설정/{n}.ply",
    "2DGS":              D + "/3_학습결과/4차_colab/out/corridor{n}/gs2d/ply/corridor{n}_2dgs_30000.ply",
    "Difix":             D + "/3_학습결과/4차_colab/out/corridor{n}/ply/corridor{n}_difix.ply",
    "Difix 위아래+보강":  D + "/3_학습결과/4차_colab/out/corridor{n}_fill/ply/corridor{n}_fill_difix.ply",
    "보강+바탕층":        D + "/3_학습결과/4차_Brush_fill/{n}.ply",
    "보강+바탕층+정리":   D + "/3_학습결과/4차_Brush_fill_clean/{n}.ply",
}
SPARSE = D + "/2_정합데이터/4차/{n}/sparse/0"
OUT = D + "/3_학습결과/비교이미지"
os.makedirs(OUT + "/tiles", exist_ok=True)
for n in SECTIONS:
    print(f"[구간 {n}]")
    for k, v in VERSIONS.items():
        print("   ", "O" if os.path.exists(v.format(n=n)) else "-", k)

## 2. 설치 (gsplat, 처음 렌더링 때 CUDA 컴파일)

In [ ]:
!pip install -q gsplat ninja
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1 || true
import torch, gsplat
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "gsplat", gsplat.__version__)

## 3. 읽기 · 렌더링 함수

In [ ]:
import struct, numpy as np, torch, math
from gsplat import rasterization
from PIL import Image, ImageDraw, ImageFont

def read_ply(path):
    with open(path, "rb") as f:
        names, n = [], 0
        while True:
            l = f.readline().decode("ascii").strip()
            if l.startswith("element vertex"): n = int(l.split()[-1])
            elif l.startswith("property"): names.append(l.split()[-1])
            elif l == "end_header": break
        A = np.frombuffer(f.read(n * len(names) * 4), "<f4").reshape(n, len(names))
    col = lambda k: torch.tensor(A[:, names.index(k)], device="cuda")
    means = torch.stack([col("x"), col("y"), col("z")], 1)
    quats = torch.stack([col(f"rot_{i}") for i in range(4)], 1); quats = quats / quats.norm(dim=1, keepdim=True)
    scales = torch.exp(torch.stack([col(f"scale_{i}") for i in range(3)], 1))
    opac = torch.sigmoid(col("opacity"))
    dc = torch.stack([col(f"f_dc_{i}") for i in range(3)], 1)[:, None, :]
    nrest = sum(1 for k in names if k.startswith("f_rest_")) // 3
    if nrest:
        rest = torch.stack([col(f"f_rest_{i}") for i in range(nrest * 3)], 1).reshape(-1, 3, nrest).transpose(1, 2)
        sh = torch.cat([dc, rest], 1)
    else:
        sh = dc
    deg = int(round(math.sqrt(sh.shape[1]))) - 1
    return dict(means=means, quats=quats, scales=scales, opac=opac, sh=sh, deg=deg, n=n)

def qvec2rot(q):
    w, x, y, z = q
    return np.array([[1-2*y*y-2*z*z, 2*x*y-2*w*z, 2*x*z+2*w*y], [2*x*y+2*w*z, 1-2*x*x-2*z*z, 2*y*z-2*w*x], [2*x*z-2*w*y, 2*y*z+2*w*x, 1-2*x*x-2*y*y]])

def read_colmap(sp):
    with open(sp + "/cameras.bin", "rb") as f:
        f.read(8); cid, model, w, h = struct.unpack("<iiQQ", f.read(24))
        npar = {0: 3, 1: 4}[model]; pp = struct.unpack("<%dd" % npar, f.read(8 * npar))
    fx, fy, cx, cy = (pp[0], pp[0], pp[1], pp[2]) if model == 0 else pp
    data = open(sp + "/images.bin", "rb").read(); off = 8; ims = []
    for _ in range(struct.unpack_from("<Q", data, 0)[0]):
        off += 4; q = struct.unpack_from("<4d", data, off); off += 32
        t = struct.unpack_from("<3d", data, off); off += 28
        e = data.index(b"\0", off); name = data[off:e].decode(); off = e + 1
        n2, = struct.unpack_from("<Q", data, off); off += 8 + n2 * 24
        ims.append((name, qvec2rot(q), np.array(t)))
    ims.sort(key=lambda a: a[0])
    data = open(sp + "/points3D.bin", "rb").read(); off = 8; pts = []
    for _ in range(min(struct.unpack_from("<Q", data, 0)[0], 300000)):
        v = struct.unpack_from("<Q3d3BdQ", data, off); off += 51 + 8 * v[-1]; pts.append(v[1:4])
    return (w, h, fx, fy, cx, cy), ims, np.array(pts)

def rot_x(a): c, s = math.cos(a), math.sin(a); return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])
def rot_y(a): c, s = math.cos(a), math.sin(a); return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def make_views(ims, pts, n_cams):
    """촬영 위치 몇 곳 x 시점 6가지 -> (이름, world->cam 4x4)"""
    C = np.array([-R.T @ t for _, R, t in ims])
    from scipy.spatial import cKDTree
    dist = np.median(cKDTree(pts).query(C)[0])           # 카메라에서 가장 가까운 표면까지 (축척 기준)
    picks = np.linspace(len(ims) * 0.1, len(ims) * 0.9, n_cams).astype(int)
    variants = [("정면", 0, 0, 0), ("왼쪽45", 45, 0, 0), ("오른쪽70", -70, 0, 0),
                ("천장", 0, 35, 0), ("바닥", 0, -35, 0), ("옆으로이동", 0, 0, 1)]
    views = []
    up = -np.mean([R.T @ [0, 1, 0] for _, R, _ in ims], 0); up /= np.linalg.norm(up)   # 촬영 카메라들의 평균 위쪽
    def rod(axis, a):   # axis 둘레로 a 만큼 회전
        k = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]])
        return np.eye(3) + math.sin(a) * k + (1 - math.cos(a)) * k @ k
    for ci, i in enumerate(picks):
        _, R, t = ims[i]; Rc = R.T; Cw = -R.T @ t
        for vn, yaw, pitch, shift in variants:
            a = math.radians(yaw)                                   # 좌우는 세계의 위쪽 축 기준으로 돌림 (화면이 기울지 않게)
            Ry = rod(up, a)
            if yaw and np.sign((Ry @ Rc[:, 2]) @ Rc[:, 0]) == np.sign(yaw): Ry = rod(up, -a)   # 양수 = 왼쪽
            R2 = Ry @ Rc @ rot_x(math.radians(pitch))
            C2 = Cw + Rc[:, 0] * shift * dist * 1.5              # 카메라 오른쪽으로 이동 (촬영 경로 밖)
            M = np.eye(4); M[:3, :3] = R2.T; M[:3, 3] = -R2.T @ C2
            views.append((ci, vn, M))
    return views

@torch.no_grad()
def render(g, M, cam, width):
    w, h, fx, fy, cx, cy = cam; s = width / w
    K = torch.tensor([[fx * s, 0, cx * s], [0, fy * s, cy * s], [0, 0, 1]], device="cuda", dtype=torch.float32)
    V = torch.tensor(M, device="cuda", dtype=torch.float32)
    img, _, _ = rasterization(g["means"], g["quats"], g["scales"], g["opac"], g["sh"], V[None], K[None],
                              width, int(round(h * s)), sh_degree=g["deg"],
                              backgrounds=torch.tensor([[1.0, 0.0, 1.0]], device="cuda"))
    return (img[0].clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)

try:
    FONT = ImageFont.truetype("/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf", 22)
except Exception:
    FONT = ImageFont.load_default()
print("준비 OK")

## 4. 렌더링 · 비교 이미지 저장

In [ ]:
import gc
for n in SECTIONS:
    cam, ims, pts = read_colmap(SPARSE.format(n=n))
    views = make_views(ims, pts, N_CAMS)
    vers = [(k, v.format(n=n)) for k, v in VERSIONS.items() if os.path.exists(v.format(n=n))]
    tiles = {}
    for name, path in vers:
        print(f"[구간 {n}] {name} 읽는 중..."); g = read_ply(path); print(f"   스플랫 {g['n']:,}")
        for ci, vn, M in views:
            im = render(g, M, cam, WIDTH); tiles[(ci, vn, name)] = im
            Image.fromarray(im).save(f"{OUT}/tiles/{n}_cam{ci}_{vn}_{name.replace(' ', '')}.png")
        del g; gc.collect(); torch.cuda.empty_cache()
    th, tw = next(iter(tiles.values())).shape[:2]
    vnames = [v[1] for v in views[:6]]
    for ci in range(N_CAMS):
        head = 40; side = 120
        sheet = Image.new("RGB", (side + tw * len(vers), head + th * len(vnames)), "white")
        d = ImageDraw.Draw(sheet)
        for j, (name, _) in enumerate(vers): d.text((side + j * tw + 8, 8), name, fill="black", font=FONT)
        for r, vn in enumerate(vnames):
            d.text((8, head + r * th + th // 2 - 12), vn, fill="black", font=FONT)
            for j, (name, _) in enumerate(vers):
                sheet.paste(Image.fromarray(tiles[(ci, vn, name)]), (side + j * tw, head + r * th))
        p = f"{OUT}/{n}_cam{ci}.png"; sheet.save(p); print("저장", p)
from IPython.display import Image as Show, display
display(Show(f"{OUT}/{SECTIONS[0]}_cam0.png", width=1400))
print("끝. 런타임 > 런타임 연결 해제 및 삭제")